# 00 — Business Domain

> **Vai trò của notebook này**: Đặt bối cảnh nghiệp vụ cho toàn bộ dự án. Đọc notebook này trước bất kỳ notebook nào khác.
>
> Không có code phân tích, không load dữ liệu, không train model.

---

## Table of Contents

1. [Mục tiêu bài toán](#1)
2. [Luồng nghiệp vụ](#2)
3. [Mô tả các bảng đầu vào](#3)
4. [Mười fraud scenario (TXN-01 → TXN-10)](#4)
5. [Phạm vi và hạn chế](#5)

---
<a id="1"></a>
## 1. Mục tiêu bài toán

Ngân hàng cần một model phát hiện **giao dịch gian lận** trong thời gian thực, trước khi tiền rời khỏi tài khoản.

### 1.1 Prediction problem

| Thuộc tính | Giá trị |
|---|---|
| **Grain** | Một dòng = một giao dịch (`transaction_id`) |
| **Thời điểm scoring** | `transaction_at` — ngay khi giao dịch được gửi đến hệ thống |
| **Target** | `target_fraud ∈ {0, 1}` — giao dịch gian lận hay hợp lệ |
| **Entity key** | `customer_id` / `account_id` — dùng khi group-split |

### 1.2 Ba population

Không phải chỉ có hai class fraud / legitimate. Dataset phân biệt ba nhóm:

| Population | Ý nghĩa | Vai trò trong modelling |
|---|---|---|
| **Confirmed fraud** | Giao dịch đã được điều tra và xác nhận gian lận | Positive class (`target_fraud = 1`) |
| **Hard-negative** | Giao dịch có tín hiệu rủi ro cao nhưng điều tra xác nhận hợp lệ | Negative, giữ riêng để đánh giá false-positive rate |
| **Background** | Giao dịch bình thường, không thuộc scenario nào | Negative, chiếm đa số |

Hard-negative quan trọng hơn background negative vì chúng kiểm tra khả năng model **không kêu nhầm** trên các case trông đáng ngờ nhưng thực tế hợp lệ.

### 1.3 Output kỳ vọng

Model trả về **risk score 0–100**. Score này sẽ được SAS Intelligent Decisioning kết hợp với business rule để ra quyết định cuối cùng: `ACCEPT`, `CHALLENGE` (yêu cầu xác thực thêm), hoặc `HOLD` (chặn giao dịch chờ điều tra).

---
<a id="2"></a>
## 2. Luồng nghiệp vụ

Giao dịch đi qua bốn giai đoạn từ lúc khách hàng thao tác đến khi hệ thống ra quyết định.

### 2.1 Pipeline tổng quan

```
┌──────────────┐     ┌───────────────────┐     ┌─────────────────┐     ┌──────────────────┐
│  Transaction  │────▶│   Feature          │────▶│   ML Model       │────▶│  SAS Intelligent  │
│  (raw event)  │     │   Enrichment       │     │   Scoring        │     │  Decisioning      │
└──────────────┘     └───────────────────┘     └─────────────────┘     └──────────────────┘
    Khách hàng            Join session,             Trả risk score         Rule + score →
    chuyển tiền           device, account,          0–100                  ACCEPT / CHALLENGE
                          beneficiary,                                     / HOLD → Alert
                          rolling features
```

### 2.2 Chi tiết từng giai đoạn

**Giai đoạn 1 — Transaction**: Khách hàng thực hiện giao dịch qua mobile/web/branch. Hệ thống nhận được message chứa thông tin cơ bản: số tiền, tài khoản, thiết bị, phiên đăng nhập, người thụ hưởng.

**Giai đoạn 2 — Feature Enrichment**: Message được làm giàu bằng cách join với dữ liệu lịch sử:
- **Account**: trạng thái, hạn mức, số dư trung bình, tuổi tài khoản.
- **Device**: fingerprint, emulator, root, trust status.
- **Session**: new device/location, VPN/proxy, thời gian đăng nhập.
- **Beneficiary**: tuổi beneficiary, risk level, cluster.
- **Rolling features**: velocity 10 phút/1 giờ/24 giờ, failed auth count, sensitive changes.

Tất cả dữ liệu enrichment phải có **trước hoặc tại thời điểm** `transaction_at`. Không được dùng thông tin tương lai.

**Giai đoạn 3 — ML Model Scoring**: Feature vector được gửi vào model. Model trả risk score 0–100.

**Giai đoạn 4 — SAS Decisioning**: Score kết hợp với business rule tạo quyết định cuối cùng. Nếu cần chặn → tạo Alert → mở Case điều tra → Verification → cập nhật Ground Truth.

### 2.3 Ranh giới model

Model chỉ tham gia **giai đoạn 3**. Các bảng thuộc giai đoạn 4 (`decision_outcomes`, `alerts`, `cases`, `verification_results`, `fraud_ground_truth`) là **kết quả** phát sinh sau model — không bao giờ là feature input. Chúng chỉ dùng để tạo label và đánh giá.

---
<a id="3"></a>
## 3. Mô tả các bảng đầu vào

Dataset gồm nhiều bảng normalized CSV, sinh bởi `run_training_raw.py` từ 5 simulation run độc lập.

Các bảng được chia thành ba nhóm theo vai trò: **Feature input**, **Label** và **Operational** (chỉ tham khảo).

### 3.1 Nhóm Feature Input — Shared Foundation

Các bảng nền tảng, dùng chung cho mọi transaction.

| Bảng | Grain | Ý nghĩa nghiệp vụ | Feature tiêu biểu |
|---|---|---|---|
| `customers` | 1 khách hàng | KYC, nhân khẩu, phân khúc, risk nền | segment, kyc_level, base_risk_level, province |
| `accounts` | 1 tài khoản | Trạng thái, hạn mức, số dư, tuổi | status, dormant_since, transfer_limit, average_balance |
| `devices` | 1 thiết bị | Fingerprint, trust, emulator, root | is_emulator, is_rooted, trust_status, device_risk_score |

### 3.2 Nhóm Feature Input — Transaction Domain

Các bảng trực tiếp liên quan đến giao dịch và hành vi xung quanh.

| Bảng | Grain | Ý nghĩa | Feature tiêu biểu |
|---|---|---|---|
| `transactions` | 1 giao dịch | Sự kiện tiền vào/ra — **bảng chính** | amount, direction, channel, balance ratio |
| `transaction_features` | 1 bộ feature/giao dịch | Rolling feature đã tính sẵn bởi generator | velocity 10m/1h, amount 24h, failed auth, sensitive change |
| `login_sessions` | 1 phiên đăng nhập | Nơi, thời gian, thiết bị, kết quả login | new_device, new_location, VPN/proxy, session_risk |
| `beneficiaries` | 1 người thụ hưởng | Danh bạ người nhận của account | tuổi beneficiary, risk level, mule cluster |
| `account_change_events` | 1 lần đổi thông tin | Đổi phone/password/device/limit | loại thay đổi, thời gian trước giao dịch |
| `auth_events` | 1 lần xác thực | OTP/password/biometric và kết quả | failed auth velocity, brute-force pattern |

### 3.3 Quan hệ giữa các bảng

Khi xây model-ready table, bắt đầu từ `transactions` rồi join ra:

```
customers ─────1:N───── accounts ─────1:N───── transactions ─────1:1───── transaction_features
                           │                       │
                           │                       ├──N:1── login_sessions ──N:1── devices
                           │                       │
                           │                       └──N:0..1── beneficiaries
                           │
                           └──1:N── account_change_events

                        auth_events ──── aggregate by account/time ──── join as rolling feature
```

**Lưu ý join**:

| Bảng | Join key từ transactions | Cardinality |
|---|---|---|
| `customers` | `customer_id` | many-to-one |
| `accounts` | `account_id` | many-to-one |
| `login_sessions` | `session_id` | many-to-one |
| `devices` | `device_id` | many-to-one |
| `beneficiaries` | `beneficiary_id` | many-to-zero/one |
| `transaction_features` | `transaction_id` | one-to-one |
| `auth_events` | aggregate trước khi join | many-to-one |
| `account_change_events` | aggregate trước khi join | many-to-one |

`auth_events` và `account_change_events` là bảng **1:N** — không join trực tiếp vào transaction vì sẽ nhân bản dòng. Phải aggregate (đếm, max, min…) theo `account_id` với điều kiện thời gian `< transaction_at` trước khi join.

### 3.4 Nhóm Label

Các bảng dùng để tạo nhãn cho model, **không phải feature input**.

| Bảng | Vai trò |
|---|---|
| `scenario_event_entities` | **Label bridge** — liên kết mỗi transaction với fraud event, vai trò (primary/supporting), và label scope (fraud/hard_negative/context_only) |
| `fraud_ground_truth` | Kết quả điều tra cuối cùng của mỗi event — dùng đối chiếu, không join trực tiếp làm feature |
| `scenario_manifest` | Metadata generator — truy vết event được tiêm, không thuộc schema production |

**Label policy**:

| Điều kiện trong bridge | `target_fraud` | `hard_negative` |
|---|---|---|
| `label_scope = fraud`, `entity_type = transaction` | 1 | 0 |
| `label_scope = hard_negative`, `entity_type = transaction` | 0 | 1 |
| `label_scope = context_only` | Exclude hoặc 0 | 0 |
| Không có trong bridge (background) | 0 | 0 |

Không được suy nhãn bằng pattern `_SCN_` trong ID — đó là dấu vết generator, không tồn tại trong production.

### 3.5 Nhóm Operational — Chỉ tham khảo

Các bảng này mô phỏng lớp vận hành **sau khi** hệ thống đã scoring. Chúng **không bao giờ** là feature input cho model.

| Bảng | Ý nghĩa | Lý do không dùng làm feature |
|---|---|---|
| `rules` | Định nghĩa business rule | Rule là logic chạy song song model, không phải input |
| `decision_outcomes` | Quyết định ACCEPT/CHALLENGE/HOLD | Kết quả **sau** scoring |
| `rule_hits` | Rule nào bắn trên decision nào | Kết quả **sau** scoring |
| `alerts` | Cảnh báo được tạo khi score/rule vượt ngưỡng | Kết quả **sau** scoring |
| `cases` | Vụ điều tra gắn với alert | Kết quả **sau** scoring |
| `verification_results` | Kết quả xác minh sau điều tra | Kết quả **sau** scoring |

---
<a id="4"></a>
## 4. Mười fraud scenario (TXN-01 → TXN-10)

Dataset mô phỏng 10 kịch bản gian lận giao dịch phổ biến tại ngân hàng Việt Nam. Mỗi scenario được "tiêm" vào dữ liệu nền bằng `scenario_engine.py`, tạo ra chuỗi hành vi mang đặc trưng fraud thật.

Ngoài 10 scenario chính, mỗi scenario còn có bản **hard-negative** (`HN-TXN-01..10`) — hành vi tương tự nhưng được xác minh là hợp lệ — và nhóm `FP-TXN` (background transaction có tín hiệu rủi ro nhưng thực tế legitimate).

### TXN-01 — Impossible Travel

**Câu chuyện**: Một tài khoản đăng nhập ở Hà Nội, 15 phút sau lại giao dịch ở TP.HCM — khoảng cách 1.700 km mà không thể di chuyển trong thời gian đó.

**Chuỗi hành vi**: Login tại địa điểm A → Login tại địa điểm B (khoảng cách lớn, thời gian ngắn) → Chuyển tiền.

| Tín hiệu | Bảng nguồn | Field |
|---|---|---|
| Khoảng cách địa lý lớn trong thời gian ngắn | `login_sessions` | `latitude`, `longitude`, `login_at` |
| Vị trí mới | `login_sessions` | `is_new_location = true` |
| Thiết bị mới (có thể) | `login_sessions` | `is_new_device = true` |

### TXN-02 — Dormant Account Awakening

**Câu chuyện**: Tài khoản không hoạt động hơn 2 năm bỗng nhiên active lại từ thiết bị mới, ngay lập tức chuyển gần hết hạn mức.

**Chuỗi hành vi**: Account dormant → Device mới đăng nhập → Thêm beneficiary → Chuyển tiền lớn.

| Tín hiệu | Bảng nguồn | Field |
|---|---|---|
| Account ở trạng thái dormant | `accounts` | `status = dormant`, `dormant_since` |
| Thiết bị mới, vị trí mới | `login_sessions` | `is_new_device`, `is_new_location` |
| Số tiền gần hạn mức | `transactions`, `accounts` | `amount` / `single_txn_limit` |

### TXN-03 — Brute-Force / Credential Stuffing

**Câu chuyện**: Kẻ tấn công thử đăng nhập nhiều lần với mật khẩu sai, cuối cùng thành công.

**Chuỗi hành vi**: Auth fail × N → Auth success → (có thể chưa có giao dịch).

| Tín hiệu | Bảng nguồn | Field |
|---|---|---|
| Nhiều auth fail liên tiếp | `auth_events` | `auth_result = failed`, `failed_attempt_count` |
| Auth risk score cao | `auth_events` | `auth_risk_score` |

> ⚠️ **Lưu ý**: TXN-03 không tạo giao dịch — chỉ có auth event gắn session. Trong label bridge, 100 dòng TXN-03 có `entity_type = account`, không phải `transaction`. Vì grain của model là transaction, TXN-03 **không đi vào training** nhưng tín hiệu failed auth vẫn dùng được như rolling feature cho các scenario khác.

### TXN-04 — Velocity Burst

**Câu chuyện**: Nhiều giao dịch nhỏ dưới ngưỡng giám sát xảy ra trong vài phút — kỹ thuật "chia nhỏ" (structuring) để tránh bị phát hiện.

**Chuỗi hành vi**: 5 giao dịch ~4,9 triệu VND mỗi cái, cách nhau 1–2 phút, cùng beneficiary.

| Tín hiệu | Bảng nguồn | Field |
|---|---|---|
| Số giao dịch cao trong 10 phút | `transaction_features` | `txn_count_10m ≥ 3` |
| Số giao dịch cao trong 1 giờ | `transaction_features` | `txn_count_1h ≥ 5` |
| Tổng tiền 24h lớn | `transaction_features` | `txn_amount_sum_24h` |

> TXN-04 tạo nhiều supporting transaction (velocity burst). Tất cả đều mang label `fraud`, không chỉ transaction cuối.

### TXN-05 — Rapid New Beneficiary Transfer

**Câu chuyện**: Nạn nhân bị lừa đảo (scam) thêm beneficiary mới rồi chuyển tiền lớn ngay lập tức — thường trong vòng vài phút.

**Chuỗi hành vi**: Thêm beneficiary mới → Chuyển tiền lớn (≈95% hạn mức) trong 1–2 phút.

| Tín hiệu | Bảng nguồn | Field |
|---|---|---|
| Beneficiary mới (< 60 phút) | `transaction_features` | `time_since_beneficiary_added_minutes ≤ 60` |
| Cờ new beneficiary | `transaction_features` | `is_new_beneficiary = true` |
| Số tiền gần hạn mức | `transactions`, `accounts` | `amount` / `single_txn_limit` ≈ 0.95 |

### TXN-06 — Full Account Takeover (ATO)

**Câu chuyện**: Kẻ tấn công chiếm quyền tài khoản — đăng nhập từ thiết bị mới qua VPN, đổi mật khẩu + số điện thoại + hạn mức, thêm beneficiary rồi rút gần hết số dư.

**Chuỗi hành vi**: Device mới + VPN → OTP verify → Đổi password → Đổi phone → Tăng limit → Thêm beneficiary → Chuyển 90% balance → Xóa beneficiary.

| Tín hiệu | Bảng nguồn | Field |
|---|---|---|
| Thiết bị mới, VPN | `login_sessions` | `is_new_device`, `vpn_flag` |
| Sensitive change gần đây | `transaction_features` | `is_after_sensitive_change`, `time_since_sensitive_change_minutes ≤ 30` |
| Nhiều loại thay đổi | `account_change_events` | `change_type` (password, phone, transfer_limit) |
| Rút gần hết balance | `transactions` | `amount` / `balance_before` > 0.8 |

> TXN-06 là scenario phức tạp nhất, kết hợp nhiều tín hiệu. Đây là lý do model cần feature kết hợp, không chỉ threshold đơn lẻ.

### TXN-07 — Money Mule Network

**Câu chuyện**: Mạng lưới tài khoản mule — tiền từ nhiều nạn nhân chảy vào tài khoản trung gian (fan-in), được phân tán qua các mule khác (layering), cuối cùng rút tiền mặt (cash-out).

**Chuỗi hành vi**: 5 nạn nhân credit vào mule A → Mule A chuyển sang mule B, C (nội bộ) → B, C rút ATM.

| Tín hiệu | Bảng nguồn | Field |
|---|---|---|
| Nhiều CREDIT đến từ nhiều nguồn (fan-in) | `transactions` | `direction = CREDIT`, count nguồn |
| Chuyển nội bộ liền sau | `transactions` | `counterparty_internal_account_id` ≠ '' |
| Mule cluster | `beneficiaries` | `mule_cluster_id` |
| Rút tiền mặt ATM | `transactions` | `transaction_type = cash_withdrawal`, `channel = atm` |

**Roles trong bridge**: `mule_inflow` (tiền vào), `layering` (chuyển tiếp), `cash_out` (rút ra), `primary` (giao dịch đại diện).

> TXN-07 tạo nhiều transaction nhất (900/run) vì mỗi ring có 5 nạn nhân + 3 mule với nhiều giao dịch.

### TXN-08 — Emulator / Proxy Bot Farm

**Câu chuyện**: Bot farm dùng emulator Android và proxy xoay vòng để brute-force đăng nhập hàng loạt tài khoản, vài tài khoản bị chiếm thành công và bị rút tiền.

**Chuỗi hành vi**: 10 tài khoản bị tấn công → 8 fail → 2 success → Thêm beneficiary → Chuyển tiền.

| Tín hiệu | Bảng nguồn | Field |
|---|---|---|
| Emulator | `devices` | `is_emulator = true` |
| Rooted/jailbroken | `devices` | `is_rooted_or_jailbroken = true` |
| Proxy | `login_sessions` | `proxy_flag = true` |
| Thiết bị mới, vị trí mới | `login_sessions` | `is_new_device`, `is_new_location` |
| Device risk score rất cao | `devices` | `device_risk_score ≥ 95` |

### TXN-09 — Rogue Employee / Internal Fund Diversion

**Câu chuyện**: Nhân viên chi nhánh dùng terminal nội bộ để chuyển tiền từ tài khoản khách hàng sang tài khoản đồng phạm, sau đó rút ngoại.

**Chuỗi hành vi**: Branch terminal → Chuyển 18 triệu nội bộ → Tài khoản nhận chuyển tiếp 17 triệu ra ngoài.

| Tín hiệu | Bảng nguồn | Field |
|---|---|---|
| Channel = branch (ngoài giờ) | `transactions` | `channel = branch`, giờ giao dịch |
| Thiết bị nội bộ | `devices` | `device_type = internal_terminal` |
| Chuyển nội bộ | `transactions` | `counterparty_internal_account_id` ≠ '' |
| Rút ra ngay sau | Giao dịch tiếp theo trên tài khoản nhận |

### TXN-10 — SIM Swap + Account Takeover

**Câu chuyện**: Kẻ tấn công lừa nhà mạng cấp lại SIM, dùng SIM mới để thay đổi thiết bị tin cậy và hạn mức, rồi rút gần hết số dư.

**Chuỗi hành vi**: (Offline SIM swap) → Đổi phone → Đổi trusted device → Tăng transfer limit → Login từ device mới → Thêm beneficiary → Chuyển 90% balance → Xóa beneficiary.

| Tín hiệu | Bảng nguồn | Field |
|---|---|---|
| Nhiều sensitive change liên tiếp | `account_change_events` | phone, trusted_device, transfer_limit |
| Thiết bị mới, risk cao | `login_sessions`, `devices` | `is_new_device`, `device_risk_score ≥ 97` |
| Rút gần hết balance | `transactions` | `amount` / `balance_before` ≈ 0.9 |
| Beneficiary bị xóa sau giao dịch | `beneficiaries` | `status = removed` |

> TXN-10 tương tự TXN-06 nhưng bắt đầu bằng SIM swap (thay đổi phone/device trước khi login), trong khi TXN-06 bắt đầu bằng credential compromise (login trước rồi mới đổi).

### Tổng hợp scenario × tín hiệu

| Scenario | Business Pattern | Tín hiệu chính |
|---|---|---|
| TXN-01 | Impossible travel | geo distance, new_location |
| TXN-02 | Dormant awakening | dormant status, new_device |
| TXN-03 | Brute-force | failed auth count (account-level, không có txn) |
| TXN-04 | Velocity burst | txn_count_10m, txn_count_1h |
| TXN-05 | Rapid new beneficiary | time_since_beneficiary ≤ 60, amt/limit |
| TXN-06 | Full ATO | sensitive_change, new_device, VPN, amt/balance |
| TXN-07 | Money mule network | fan-in/out, mule_cluster, cash withdrawal |
| TXN-08 | Bot farm | emulator, proxy, new_device |
| TXN-09 | Rogue employee | branch channel, internal transfer, night hour |
| TXN-10 | SIM swap | sensitive_change chain, new_device, amt/balance |

---
<a id="5"></a>
## 5. Phạm vi và hạn chế

### 5.1 Trong phạm vi

- **Chỉ transaction fraud** — loan domain (LOAN-01..10) không nằm trong scope.
- Model dự đoán tại thời điểm giao dịch (`transaction_at`).
- Dữ liệu từ 5 simulation run, tổng ~112.500 giao dịch.
- Target = fraud/legitimate theo `scenario_event_entities` bridge.

### 5.2 Ngoài phạm vi

- Loan application fraud (LOAN-01..10) — cần notebook và model riêng.
- Business rule engine — rules được định nghĩa trong SAS Decisioning, model chỉ cung cấp score.
- Alert/case/verification workflow — đây là lớp vận hành sau scoring.
- Real-time serving architecture — tham khảo `docs/sas_model_scoring_architecture.md`.

### 5.3 Hạn chế của dữ liệu synthetic

| Hạn chế | Ảnh hưởng |
|---|---|
| Tất cả transaction đều `status = success` | Cột status không phân biệt, cần drop |
| Hầu hết là transfer; chưa có card/merchant | Feature merchant_id, MCC trống 100% |
| Tín hiệu fraud được set rõ ràng bởi generator | Model có thể học shortcut — cần ablation test |
| Operational scores tạo từ số rule kỳ vọng | Không phải score độc lập — không dùng làm feature |
| Ground truth là event-level | Chưa liệt kê đầy đủ mọi entity tham gia event |
| Seed flag (`is_synthetic_identity_seed`, `is_mule_candidate_seed`) | Điều khiển generator, không tồn tại trong production — PHẢI drop |

### 5.4 Lộ trình notebook tiếp theo

| Notebook | Nội dung |
|---|---|
| **01 — Data Quality** | Schema check, PK/FK, missing, timeline, balance chain |
| **02 — EDA** | Phân phối feature, target-aware analysis, univariate/bivariate |
| **03 — Feature Engineering** | Point-in-time feature mart, leakage audit, allow/deny list |
| **04 — Modelling** | Baseline rule, logistic regression, GBM, evaluation |